# SecureSense - Fixed Train/Test Split

## Objective

This notebook creates the fixed datasets used for evaluating all SecureSense
machine-learning and deep-learning models.

The cleaned CICIDS2017 dataset has already undergone:

- Missing-value validation
- Infinite-value validation
- Constant-feature removal
- Binary-label conflict removal
- Exact duplicate removal

The final dataset contains:

- 2,231,266 network-flow records
- 69 predictive features
- BENIGN and ATTACK binary labels

## Important Evaluation Principle

The same test dataset will be used for every model.

This ensures that Logistic Regression, Decision Tree, Random Forest,
HistGradientBoosting, and ANN are compared fairly.

The test data will not be used during model training or preprocessing.

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

DATA_PATH = "../data/cleaned/securesense_final_binary.parquet"

OUTPUT_DIR = "../data/splits"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Cleaned Dataset

The cleaned dataset generated by the SecureSense preprocessing pipeline
is loaded here.

The `_source_file` column is retained for auditing and later
generalization experiments, but it will NEVER be used as a model feature.

In [2]:
df = pd.read_parquet(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

df.head()

Dataset loaded successfully.
Rows: 2,231,266
Columns: 72


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,_source_file,BinaryLabel
0,6,4,2,0,12,0,6,6,6.00000,0.000000,...,0.0,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,BENIGN
1,6,1,2,0,12,0,6,6,6.00000,0.000000,...,0.0,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,BENIGN
2,6,3,2,0,12,0,6,6,6.00000,0.000000,...,0.0,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,BENIGN
3,6,1,2,0,12,0,6,6,6.00000,0.000000,...,0.0,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,BENIGN
4,6,609,7,4,484,414,233,0,69.14286,111.967896,...,0.0,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,BENIGN


## 2. Binary Class Distribution

SecureSense performs binary cyber-threat detection:

- BENIGN → normal network activity
- ATTACK → potentially malicious network activity

In [3]:
print("Binary label distribution:")
print(df["BinaryLabel"].value_counts())

print("\nBinary label percentages:")
print(
    df["BinaryLabel"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Binary label distribution:
BinaryLabel
BENIGN    1895044
ATTACK     336222
Name: count, dtype: int64

Binary label percentages:
BinaryLabel
BENIGN    84.93
ATTACK    15.07
Name: proportion, dtype: float64


## 3. Separate Features and Target

The model must never receive information about the original dataset source.

Therefore:

- `BinaryLabel` is the target.
- `_source_file` is retained only for dataset auditing.
- `Label` is the original multiclass CICIDS2017 label and is not used as an input feature.

In [4]:
EXCLUDED_COLUMNS = [
    "BinaryLabel",
    "Label",
    "_source_file"
]

FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in EXCLUDED_COLUMNS
]

X = df[FEATURE_COLUMNS]
y = df["BinaryLabel"]

print(f"Number of model features: {len(FEATURE_COLUMNS)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Number of model features: 69
X shape: (2231266, 69)
y shape: (2231266,)


In [5]:
y_encoded = y.map({
    "BENIGN": 0,
    "ATTACK": 1
})

if y_encoded.isna().any():
    raise ValueError("Unexpected label found.")

y_encoded = y_encoded.astype(np.int8)

print(y_encoded.value_counts())

BinaryLabel
0    1895044
1     336222
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print("Training set:")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print("\nTest set:")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

Training set:
X_train: (1785012, 69)
y_train: (1785012,)

Test set:
X_test: (446254, 69)
y_test: (446254,)


In [7]:
print("Training class distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTest class distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training class distribution:
BinaryLabel
0    84.93
1    15.07
Name: proportion, dtype: float64

Test class distribution:
BinaryLabel
0    84.93
1    15.07
Name: proportion, dtype: float64


In [8]:
train_df = X_train.copy()
train_df["BinaryLabel"] = y_train.values

test_df = X_test.copy()
test_df["BinaryLabel"] = y_test.values

train_path = os.path.join(
    OUTPUT_DIR,
    "train.parquet"
)

test_path = os.path.join(
    OUTPUT_DIR,
    "test.parquet"
)

train_df.to_parquet(
    train_path,
    index=False
)

test_df.to_parquet(
    test_path,
    index=False
)

print("Fixed datasets saved successfully.")

print(f"\nTraining file:")
print(train_path)

print(f"\nTesting file:")
print(test_path)

Fixed datasets saved successfully.

Training file:
../data/splits\train.parquet

Testing file:
../data/splits\test.parquet


In [9]:
print("=" * 70)
print("FINAL SPLIT VALIDATION")
print("=" * 70)

print(f"Training rows: {len(train_df):,}")
print(f"Testing rows:  {len(test_df):,}")

print(f"\nFeatures: {len(FEATURE_COLUMNS)}")

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

print("\nMissing values in training:")
print(X_train.isna().sum().sum())

print("\nMissing values in testing:")
print(X_test.isna().sum().sum())

print("\n" + "=" * 70)
print("FIXED SPLIT CREATION COMPLETE")
print("=" * 70)

FINAL SPLIT VALIDATION
Training rows: 1,785,012
Testing rows:  446,254

Features: 69

Training labels:
BinaryLabel
0    1516035
1     268977
Name: count, dtype: int64

Testing labels:
BinaryLabel
0    379009
1     67245
Name: count, dtype: int64

Missing values in training:
0

Missing values in testing:
0

FIXED SPLIT CREATION COMPLETE
